# Numpy

Материалы:
* Макрушин С.В. "Лекция 1: Библиотека Numpy"
* https://numpy.org/doc/stable/user/index.html
* https://numpy.org/doc/stable/reference/index.html

In [1]:
import numpy as np
from contourpy.util.data import simple
from scipy.ndimage import minimum_filter1d
from soupsieve.util import lower
from sympy import uppergamma
from sympy.benchmarks.bench_discrete_log import data_set_1
from sympy.parsing.sympy_parser import rationalize
from sympy.physics.quantum.shor import ratioize
from sympy.physics.units.definitions.unit_definitions import meganewton

## Задачи для совместного разбора

1. Сгенерировать двухмерный массив `arr` размерности (4, 7), состоящий из случайных действительных чисел, равномерно распределенных в диапазоне от 0 до 20. Нормализовать значения массива с помощью преобразования вида  $𝑎𝑥+𝑏$  так, что после нормализации максимальный элемент масcива будет равен 1.0, минимальный 0.0

2. Создать матрицу 8 на 10 из случайных целых (используя модуль `numpy.random`) чисел из диапозона от 0 до 10 и найти в ней строку (ее индекс и вывести саму строку), в которой сумма значений минимальна.

3. Найти евклидово расстояние между двумя одномерными векторами одинаковой размерности.

4. Решить матричное уравнение `A*X*B=-C` - найти матрицу `X`. Где `A = [[-1, 2, 4], [-3, 1, 2], [-3, 0, 1]]`, `B=[[3, -1], [2, 1]]`, `C=[[7, 21], [11, 8], [8, 4]]`.

## Лабораторная работа №1

Замечание: при решении данных задач не подразумевается использования циклов или генераторов Python, если в задании не сказано обратного. Решение должно опираться на использования функционала библиотеки `numpy`.

1. Файл `minutes_n_ingredients.csv` содержит информацию об идентификаторе рецепта, времени его выполнения в минутах и количестве необходимых ингредиентов. Считайте данные из этого файла в виде массива `numpy` типа `int32`, используя `np.loadtxt`. Выведите на экран первые 5 строк массива.

In [2]:
data = np.loadtxt("./data/minutes_n_ingredients.csv", skiprows=1, dtype=np.int32, delimiter=",")

In [3]:
five = data[:5]
five

array([[127244,     60,     16],
       [ 23891,     25,      7],
       [ 94746,     10,      6],
       [ 67660,      5,      6],
       [157911,     60,     14]], dtype=int32)

2. Вычислите среднее значение, минимум, максимум и медиану по каждому из столбцов, кроме первого.

In [4]:
sums_data = data.sum(axis=0)
len_data = len(data)
print(f"Средние значения столбцов 2 и 3 - {np.array(sums_data[1:] / len_data, dtype=np.int32)}")


Средние значения столбцов 2 и 3 - [21601     9]


In [5]:
_, max1, max2 = np.max(data, axis=0)
max1, max2 = max1.item(), max2.item()
print(f"Максимальное значение второго и третьего столбца - {max1, max2}")

Максимальное значение второго и третьего столбца - (2147483647, 39)


In [6]:
_, min1, min2 = np.min(data, axis=0)
min1, min2 = min1.item(), min2.item()
print(f"Минимальное значение второго и третьего столбца - {min1, min2}")

Минимальное значение второго и третьего столбца - (0, 1)


In [7]:
_, median1, median2 = np.median(data, axis=0)
median1, median2 = median1.item(), median2.item()
print(f"Медианные значения для 2 и 3 столбца - {median1, median2}")

Медианные значения для 2 и 3 столбца - (40.0, 9.0)


3. Ограничьте сверху значения продолжительности выполнения рецепта значением квантиля $q_{0.75}$.

In [8]:
time_to_cook = data[:, 1]
q75 = np.quantile(time_to_cook, 0.75)
clipped_time = np.clip(time_to_cook, a_min=None ,a_max=q75)
clipped_time

array([60., 25., 10., ..., 65.,  5., 65.], shape=(100000,))

4. Посчитайте, для скольких рецептов указана продолжительность, равная нулю. Замените для таких строк значение в данном столбце на 1.

In [9]:
time_to_cook = data[:, 1]
q1 = np.float64(1)
clipped_time = np.clip(time_to_cook, a_min=q1, a_max=np.inf)
np.min(clipped_time)

np.float64(1.0)

5. Посчитайте, сколько уникальных рецептов находится в датасете.

In [10]:
id_receipt = data[:, 0]
len(np.unique(id_receipt))

100000

6. Сколько и каких различных значений кол-ва ингредиентов присутвует в рецептах из датасета?

In [11]:
count_ingredient = data[:, 2]
len(np.unique(count_ingredient)), np.unique(count_ingredient)

(37,
 array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
        35, 37, 39], dtype=int32))

7. Создайте версию массива, содержащую информацию только о рецептах, состоящих не более чем из 5 ингредиентов.

In [12]:
count_ingredient = data[:, 2]
less5 = count_ingredient[count_ingredient <= 5]
less5

array([5, 3, 4, ..., 5, 4, 4], shape=(17262,), dtype=int32)

8. Для каждого рецепта посчитайте, сколько в среднем ингредиентов приходится на одну минуту рецепта. Найдите максимальное значение этой величины для всего датасета

In [13]:
minutes = data[:, 1]
ingridients = data[:, 2]
valid = minutes > 0

ration = ingridients[valid] / minutes[valid]
max(ration)

np.float64(23.0)

9. Вычислите среднее количество ингредиентов для топ-100 рецептов с наибольшей продолжительностью

In [16]:
minutes = data[:, 1]
ingridients = data[:, 2]

sorted_min = np.argsort(minutes)[-100:]
sorted_ingridients = ingridients[sorted_min]
mean = np.mean(sorted_ingridients)
mean

np.float64(6.61)

10. Выберите случайным образом и выведите информацию о 10 различных рецептах

In [21]:
d = np.random.choice(len(data[:,0]), size=10, replace=False)
rand = data[d]
rand

array([[342762,     30,      9],
       [390878,     40,      6],
       [ 72145,     50,     15],
       [256489,     45,      7],
       [ 18138,     25,      4],
       [299993,     40,      8],
       [210033,     20,      4],
       [109839,     20,      4],
       [100201,      7,      6],
       [ 19972,     43,     13]], dtype=int32)

11. Выведите процент рецептов, кол-во ингредиентов в которых меньше среднего.

In [26]:
ingridients = data[:, 2]
mean = np.mean(ingridients)
count = np.sum(ingridients < mean)
p = count / ingridients.size * 100
int(p.item())

58

12. Назовем "простым" такой рецепт, длительность выполнения которого не больше 20 минут и кол-во ингредиентов в котором не больше 5. Создайте версию датасета с дополнительным столбцом, значениями которого являются 1, если рецепт простой, и 0 в противном случае.

In [28]:
ingridients = data[:, 2]
time_to_cook = data[:, 1]
simple = (time_to_cook < 20) & (ingridients < 5)
simple = simple.astype(int)
data_s = np.column_stack((data, simple))
data_s

array([[127244,     60,     16,      0],
       [ 23891,     25,      7,      0],
       [ 94746,     10,      6,      0],
       ...,
       [498432,     65,     15,      0],
       [370915,      5,      4,      1],
       [ 81993,    140,     14,      0]], shape=(100000, 4))

13. Выведите процент "простых" рецептов в датасете

In [32]:
ingridients = data[:, 2]
time_to_cook = data[:, 1]
simple = (time_to_cook < 20) & (ingridients < 5)
count = np.count_nonzero(simple)
(count / data.size * 100).item()

1.6976666666666667

14. Разделим рецепты на группы по следующему правилу. Назовем рецепты короткими, если их продолжительность составляет менее 10 минут; стандартными, если их продолжительность составляет более 10, но менее 20 минут; и длинными, если их продолжительность составляет не менее 20 минут. Создайте трехмерный массив, где нулевая ось отвечает за номер группы (короткий, стандартный или длинный рецепт), первая ось - за сам рецепт и вторая ось - за характеристики рецепта. Выберите максимальное количество рецептов из каждой группы таким образом, чтобы было возможно сформировать трехмерный массив. Выведите форму полученного массива.